# Iteration runs

This notebook automates the testing of different descriptor and selector combinations. 

The same test set and validation set are used for all models. After these are removed from the dataset, the different sampling methods sample the training set from the remaining pool.

It reads from `data/settings/iteration_settings.csv` containing pairs of descriptors and selectors and their kwargs. Results are output to `outputs/results/iteration_results.csv`.

As comparisons, it trains and tests the base model and full high-fidelity model as well.

## 0.1. Imports and load data

In [17]:
import ase.io
import os
from pathlib import Path
import numpy as np
import importlib
import torch
import matplotlib.pyplot as plt
import pandas as pd
import ast
from sklearn.model_selection import train_test_split

from mace import modules
from mace.data.atom_data_loader import AtomDataLoaderBuilder
from mace.testing import Tester
from mace.training import FreezeStrategy, NaiveStrategy, Trainer, initialise_autoencoder

import sampling_methods.descriptors as descriptors
import sampling_methods.selectors as selectors
import utils.training as training

importlib.reload(descriptors)
importlib.reload(selectors)
importlib.reload(training)


<module 'utils.training' from '/home/lim_yt/X-MACE-sampling/utils/training.py'>

In [18]:
MAX_EPOCHS = 10
R_MAX = 5.0
BATCH_SIZE = 16
BASE_LR = 1.0e-3
TRANSFER_LR = 5.0e-4
DEVICE = torch.device("cpu")

# define wrapper classes
trainer = Trainer(
    max_epochs=MAX_EPOCHS, early_stopping=True, patience=15,
    restore_best=True, device=DEVICE, verbose=True,
)
data_builder = AtomDataLoaderBuilder(
    cutoff=R_MAX, energy_key="REF_energy", forces_key="REF_forces"
)
tester = Tester(device=DEVICE)
loss_fn = modules.InvariantsWeightedEnergyForcesNacsDipoleLoss(
    energy_weight=1.0, forces_weight=5.0, dipoles_weight=0.0,
    nacs_weight=0.0, socs_weight=0.0,
).to(DEVICE)


In [19]:
ROOT_PATH = Path.cwd()
DATA_DIR = ROOT_PATH / '../data'
SETTINGS_CSV = DATA_DIR / 'settings' / 'iteration_settings.csv'
OUTPUT_DIR = ROOT_PATH / '../outputs'
RESULTS_CSV = OUTPUT_DIR / 'results' / 'vary_n_clusters_results.csv'

SEED = 42
torch.manual_seed(SEED)

# Load descriptor and selector pairs from CSV
settings = pd.read_csv(SETTINGS_CSV)

# Prepare to store results
results = []

# Load base dataset
BASE_XYZ = DATA_DIR / 'A02_propene_grid_static.xyz'
BASE_N_GEOMETRIES = '500' 
base_atoms_list = ase.io.read(BASE_XYZ, index=f":{BASE_N_GEOMETRIES}")

# Load transfer dataset
TRANSFER_XYZ = DATA_DIR / "casscf_44_propene_full.xyz"
TRANSFER_N_GEOMETRIES = '500'
transfer_atoms_list = ase.io.read(TRANSFER_XYZ, index=f":{TRANSFER_N_GEOMETRIES}")


## 0.2. Split into test, train, valid sets

In [20]:
# get bond lengths and dihedrals
# tested on propene only

desc_matrix = []
bond_lengths = []
dihedrals = []

for atom in base_atoms_list:
    bond_length = descriptors.get_descriptor("bond_lengths",atom)[0]
    dihedral = descriptors.get_descriptor("dihedral",atom)[0]
    
    bond_lengths.append(bond_length)
    dihedrals.append(dihedral)
    desc_matrix.append([bond_length, dihedral])

desc_matrix = np.asarray(desc_matrix)

print("Number of unique bond lengths:", len(np.unique(bond_lengths)))
print("Number of unique dihedral angles:", len(np.unique(dihedrals)))


Number of unique bond lengths: 41
Number of unique dihedral angles: 90


In [21]:
# extract test set
# the same test set is removed from both base and transfer datasets

TEST_SET_FRACTION = 0.1
TEST_SET_SIZE = int(np.floor(int(BASE_N_GEOMETRIES) * TEST_SET_FRACTION))

test_set_idx = selectors.get_selector("uniform_grid", desc_matrix, TEST_SET_SIZE)
base_test_set = [base_atoms_list[i] for i in test_set_idx]
transfer_test_set = [transfer_atoms_list[i] for i in test_set_idx]
print("Test set size:", len(base_test_set))

# remaining geometries are for training and validation
train_valid_set_idx = np.setdiff1d(np.arange(len(transfer_atoms_list)), test_set_idx)
base_train_valid_set = [base_atoms_list[i] for i in train_valid_set_idx]
transfer_train_valid_set = [transfer_atoms_list[i] for i in train_valid_set_idx]


Test set size: 49


In [22]:
# split remaining geometries into train and valid sets
# the same for both base and transfer datasets

SEED = 42 # set as int to get the same split every time
VALID_SET_FRACTION = 0.1 # as a fraction of the total dataset

train_set_idx, valid_set_idx = train_test_split(
    train_valid_set_idx, test_size=VALID_SET_FRACTION/(1-TEST_SET_FRACTION), random_state=SEED, shuffle=True
)

base_train_set = [base_atoms_list[i] for i in train_set_idx]
print("Base train set size:", len(base_train_set))
base_valid_set = [base_atoms_list[i] for i in valid_set_idx]
print("Base valid set size:", len(base_valid_set))

transfer_train_set = [transfer_atoms_list[i] for i in train_set_idx]
print("\nFull high-fidelity train set size:", len(transfer_train_set))
transfer_valid_set = [transfer_atoms_list[i] for i in valid_set_idx]
print("Full high-fidelity valid set size:", len(transfer_valid_set))


Base train set size: 400
Base valid set size: 51

Full high-fidelity train set size: 400
Full high-fidelity valid set size: 51


## 0.3. Train base and full high-fidelity models once

In [23]:
# base model

base_train_loader = data_builder.load(
    base_train_set, batch_size=BATCH_SIZE, shuffle=True
)
base_valid_loader = data_builder.load(
    base_valid_set, batch_size=BATCH_SIZE, shuffle=False
)
base_test_loader = data_builder.load(
    base_test_set, batch_size=BATCH_SIZE, shuffle=False
)
torch.manual_seed(SEED)

base_model = initialise_autoencoder(data_builder.get_metadata(), preset="lightweight")
base_optimizer = torch.optim.Adam(base_model.parameters(), lr=BASE_LR)

base_model, base_history = trainer.train_model(
    base_model, base_train_loader, base_valid_loader, base_optimizer, loss_fn
)

tester.run_test(base_model, base_test_loader)
base_energy_mae = tester.get_energy_mae()
{"best_epoch": base_history["best_epoch"], "test_energy_mae_ev": base_energy_mae}

results.append({'descriptor': 'base', 'selector': 'base', 'best_epoch': base_history["best_epoch"], 'energy_mae': base_energy_mae})


/home/lim_yt/micromamba/envs/xmace311/lib/python3.11/ast.py:418: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  return visitor(node)
/home/lim_yt/micromamba/envs/xmace311/lib/python3.11/ast.py:418: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  return visitor(node)
/home/lim_yt/micromamba/envs/xmace311/lib/python3.11/ast.py:418: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  return visitor(node)
/home/lim_yt/micromamba/envs/xmace311/lib/python3

Epoch 001 | train_loss=110.805522 | valid_loss=143.645348
Epoch 002 | train_loss=98.711857 | valid_loss=123.440926
Epoch 003 | train_loss=59.361273 | valid_loss=40.050778
Epoch 004 | train_loss=36.559696 | valid_loss=30.456585
Epoch 005 | train_loss=34.779711 | valid_loss=26.838164
Epoch 006 | train_loss=30.554280 | valid_loss=25.200441
Epoch 007 | train_loss=28.726743 | valid_loss=25.720980
Epoch 008 | train_loss=27.143945 | valid_loss=23.137841
Epoch 009 | train_loss=26.230271 | valid_loss=22.065236
Epoch 010 | train_loss=24.822448 | valid_loss=23.055558


In [24]:
# full high-fidelity model

full_train_loader = data_builder.load(
    transfer_train_set, batch_size=BATCH_SIZE, shuffle=True
)
full_valid_loader = data_builder.load(
    transfer_valid_set, batch_size=BATCH_SIZE, shuffle=False
)
full_test_loader = data_builder.load(
    transfer_test_set, batch_size=BATCH_SIZE, shuffle=False
)
torch.manual_seed(SEED)

full_model = initialise_autoencoder(data_builder.get_metadata(), preset="lightweight")
full_optimizer = torch.optim.Adam(full_model.parameters(), lr=BASE_LR)

full_model, full_history = trainer.train_model(
    full_model, full_train_loader, full_valid_loader, full_optimizer, loss_fn
)

tester.run_test(full_model, full_test_loader)
full_energy_mae = tester.get_energy_mae()
{"best_epoch": full_history["best_epoch"], "test_energy_mae_ev": full_energy_mae}

results.append({'descriptor': 'full', 'selector': 'full', 'best_epoch': full_history["best_epoch"], 'energy_mae': full_energy_mae})


/home/lim_yt/micromamba/envs/xmace311/lib/python3.11/ast.py:418: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  return visitor(node)
/home/lim_yt/micromamba/envs/xmace311/lib/python3.11/ast.py:418: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  return visitor(node)
/home/lim_yt/micromamba/envs/xmace311/lib/python3.11/ast.py:418: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  return visitor(node)
/home/lim_yt/micromamba/envs/xmace311/lib/python3

Epoch 001 | train_loss=94.468593 | valid_loss=137.512264
Epoch 002 | train_loss=81.933370 | valid_loss=105.757315
Epoch 003 | train_loss=39.982557 | valid_loss=29.542008
Epoch 004 | train_loss=23.265453 | valid_loss=17.644935
Epoch 005 | train_loss=16.161132 | valid_loss=12.668763
Epoch 006 | train_loss=15.608734 | valid_loss=21.415628
Epoch 007 | train_loss=14.925137 | valid_loss=11.056147
Epoch 008 | train_loss=13.546927 | valid_loss=13.045257
Epoch 009 | train_loss=11.296221 | valid_loss=10.631888
Epoch 010 | train_loss=10.313873 | valid_loss=10.687597


# 1. Iterative testing

In [25]:
# functions for parsing descriptor kwargs and selector kwargs in settings file

def parse_kwargs(value):
    if pd.isna(value) or value is None:
        return {}
    if isinstance(value, str):
        value = value.strip()
        if not value:
            return {}
        try:
            parsed = ast.literal_eval(value)
        except (ValueError, SyntaxError):
            return {}
        return parsed if isinstance(parsed, dict) else {}
    return value if isinstance(value, dict) else {}

base_encoder = base_model.perm_encoder
def resolve_kwargs(kwargs):
    resolved = {}
    for k, v in kwargs.items():
        if v == "base_encoder":
            resolved[k] = base_encoder
        else:
            resolved[k] = v
    return resolved

def serialize_kwargs(kwargs):
    return {
        k: "base_encoder" if v is base_encoder else v
        for k, v in kwargs.items()
    }


In [26]:
# Initialise valid and test loaders
full_valid_loader = data_builder.load(
    transfer_valid_set, batch_size=BATCH_SIZE, shuffle=False
)
full_test_loader = data_builder.load(
    transfer_test_set, batch_size=BATCH_SIZE, shuffle=False
)

# Iterate over descriptor and selector pairs
for index, row in settings.iterrows():
    descriptor = row['descriptor']
    selector = row['selector']

    descriptor_kwargs = resolve_kwargs(parse_kwargs(row.get('descriptor_kwargs', '')))
    selector_kwargs = resolve_kwargs(parse_kwargs(row.get('selector_kwargs', '')))
    
    # Run descriptor
    desc_matrix = []
    for atom in base_train_set:
        try:
            desc = descriptors.get_descriptor(descriptor, atom, **descriptor_kwargs)
        except (TypeError, ValueError):
            desc = descriptors.get_descriptor(descriptor, atom)
        desc_matrix.append(desc)

    desc_matrix = np.asarray(desc_matrix)

    # Run selector
    try:
        sampled_idx = selectors.get_selector(selector, desc_matrix, 100, **selector_kwargs)
    except (TypeError, ValueError):
        sampled_idx = selectors.get_selector(selector, desc_matrix, 100)

    sampled_atoms = [transfer_train_set[i] for i in sampled_idx]

    # Initialise train loader
    transfer_train_loader = data_builder.load(
        sampled_atoms, batch_size=BATCH_SIZE, shuffle=True
    )

    # Train the model
    transfer_model = NaiveStrategy().apply(base_model)
    transfer_optimizer = torch.optim.Adam(transfer_model.parameters(), lr=TRANSFER_LR)

    torch.manual_seed(SEED)

    transfer_model, transfer_history = trainer.train_model(
        transfer_model, transfer_train_loader, full_valid_loader, transfer_optimizer, loss_fn
    )
    
    # Test the model
    tester = Tester(device=torch.device('cpu'))
    tester.run_test(transfer_model, full_test_loader)
    best_epoch = transfer_history["best_epoch"]
    energy_mae = tester.get_energy_mae()
    
    results.append({
        'descriptor': descriptor,
        'selector': selector,
        'descriptor_kwargs': serialize_kwargs(descriptor_kwargs),
        'selector_kwargs': serialize_kwargs(selector_kwargs),
        'best_epoch': best_epoch,
        'energy_mae': energy_mae
    })


/home/lim_yt/micromamba/envs/xmace311/lib/python3.11/site-packages/dscribe/core/system.py:96: FutureWarning: Please use atoms.calc
  calculator=atoms.get_calculator(),


n clusters: 1
Epoch 001 | train_loss=10.784671 | valid_loss=75.733775
Epoch 002 | train_loss=8.150772 | valid_loss=14.452056
Epoch 003 | train_loss=7.466882 | valid_loss=15.778805
Epoch 004 | train_loss=6.817930 | valid_loss=15.667525
Epoch 005 | train_loss=6.816014 | valid_loss=12.042418
Epoch 006 | train_loss=6.524714 | valid_loss=18.441131
Epoch 007 | train_loss=6.344418 | valid_loss=15.581732
Epoch 008 | train_loss=6.106866 | valid_loss=17.421521
Epoch 009 | train_loss=6.257033 | valid_loss=15.923841
Epoch 010 | train_loss=6.217742 | valid_loss=28.777958


/home/lim_yt/micromamba/envs/xmace311/lib/python3.11/site-packages/dscribe/core/system.py:96: FutureWarning: Please use atoms.calc
  calculator=atoms.get_calculator(),


n clusters: 5
Epoch 001 | train_loss=18.430460 | valid_loss=14.608287
Epoch 002 | train_loss=12.757809 | valid_loss=12.004302
Epoch 003 | train_loss=12.169782 | valid_loss=11.825485
Epoch 004 | train_loss=10.373248 | valid_loss=10.500314
Epoch 005 | train_loss=9.797607 | valid_loss=10.045360
Epoch 006 | train_loss=9.300674 | valid_loss=10.036284
Epoch 007 | train_loss=9.864712 | valid_loss=9.999802
Epoch 008 | train_loss=9.088443 | valid_loss=9.390040
Epoch 009 | train_loss=9.232513 | valid_loss=9.564815
Epoch 010 | train_loss=8.937265 | valid_loss=9.317349


/home/lim_yt/micromamba/envs/xmace311/lib/python3.11/site-packages/dscribe/core/system.py:96: FutureWarning: Please use atoms.calc
  calculator=atoms.get_calculator(),


...

n clusters: 90
Epoch 001 | train_loss=17.249322 | valid_loss=13.928665
Epoch 002 | train_loss=14.841922 | valid_loss=10.820586
Epoch 003 | train_loss=12.815001 | valid_loss=12.161658
Epoch 004 | train_loss=12.899243 | valid_loss=10.889452
Epoch 005 | train_loss=11.872188 | valid_loss=11.473448
Epoch 006 | train_loss=11.254323 | valid_loss=10.058427
Epoch 007 | train_loss=10.171339 | valid_loss=9.816155
Epoch 008 | train_loss=10.317473 | valid_loss=10.027584
Epoch 009 | train_loss=9.939652 | valid_loss=10.107302
Epoch 010 | train_loss=10.364173 | valid_loss=9.193987


/home/lim_yt/micromamba/envs/xmace311/lib/python3.11/site-packages/dscribe/core/system.py:96: FutureWarning: Please use atoms.calc
  calculator=atoms.get_calculator(),


n clusters: 95
Epoch 001 | train_loss=18.312696 | valid_loss=14.875724
Epoch 002 | train_loss=14.411733 | valid_loss=10.989125
Epoch 003 | train_loss=11.794186 | valid_loss=10.242801
Epoch 004 | train_loss=10.795045 | valid_loss=10.376034
Epoch 005 | train_loss=11.487397 | valid_loss=10.431009
Epoch 006 | train_loss=11.146140 | valid_loss=11.491990
Epoch 007 | train_loss=11.129283 | valid_loss=9.818906
Epoch 008 | train_loss=9.901651 | valid_loss=9.222981
Epoch 009 | train_loss=10.029303 | valid_loss=9.205153
Epoch 010 | train_loss=9.524206 | valid_loss=9.102856


/home/lim_yt/micromamba/envs/xmace311/lib/python3.11/site-packages/dscribe/core/system.py:96: FutureWarning: Please use atoms.calc
  calculator=atoms.get_calculator(),


n clusters: 100
Epoch 001 | train_loss=18.090282 | valid_loss=14.697146
Epoch 002 | train_loss=14.021085 | valid_loss=11.176347
Epoch 003 | train_loss=11.659863 | valid_loss=10.476327
Epoch 004 | train_loss=12.049468 | valid_loss=11.755044
Epoch 005 | train_loss=11.375087 | valid_loss=10.217745
Epoch 006 | train_loss=11.148035 | valid_loss=9.807752
Epoch 007 | train_loss=10.358748 | valid_loss=9.552884
Epoch 008 | train_loss=10.062206 | valid_loss=9.867672
Epoch 009 | train_loss=9.750841 | valid_loss=9.231717
Epoch 010 | train_loss=9.727397 | valid_loss=9.037454


In [27]:
# Save results to CSV
results_df = pd.DataFrame(results)
results_df.to_csv(RESULTS_CSV, index=False)
print(f'Results saved to {RESULTS_CSV}')

Results saved to /home/lim_yt/X-MACE-sampling/notebooks/../outputs/results/vary_n_clusters_results.csv
